# Primer: environments, kernels and Python essentials

**Everyone** should read the opening **Environments and kernels** section before Session 1, even if Python syntax is already familiar. Work through the remaining syntax, array and memory-view sections if you are newer to Python or moving from MATLAB, IDL, R or Mathematica.

Run cells top to bottom with **Shift+Enter** (`⏻` **Start compute** in the book toolbar, or in JupyterLab), predict each output before running, and verify finished work with **Kernel → Restart Kernel and Run All Cells**.

## Environments and kernels: pip, conda and uv

- **Why isolate & solve dependencies:** Scientific libraries form a constraint graph (for example, `astropy` and `scipy` requiring compatible `numpy` bounds). A **dependency solver (resolver)** searches the full graph for one mutually consistent set of package versions before installing:
  - Ad-hoc `pip install pkg` resolves only against whatever is currently installed (with no lockfile by default), so adding a package months later can silently upgrade `numpy` and break working research code.
  - **`uv`** *(used here)* resolves the full `pyproject.toml` graph in milliseconds and pins exact versions plus cryptographic hashes in `uv.lock` (`uv sync --frozen --extra teaching`).
  - **`conda` / `mamba`** (`libmamba` SAT solver) resolves Python packages **jointly with compiled C/Fortran/CUDA libraries** (`CFITSIO`, `FFTW`, MPI/CUDA stacks on `conda-forge`).

![Diagram: Python environments, package managers and Jupyter kernels](../book/diagrams/python-environments.svg)

**One environment per project, registered as a named kernel** ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/python-environments.dot)).

- **`Kernel != terminal` (and hidden notebook state):**
  - A notebook kernel runs one `sys.executable` (`/opt/course/.venv/bin/python` on CPD), whereas opening a terminal shell does **not** activate that environment automatically — run `source .../activate` or `uv run` first.
  - Never `pip install` into system Python or Conda `(base)`. In a `uv` project use `uv add` / `uv sync` (`uv` omits `pip` from `.venv` by default; in standard `venv`s use `python -m pip`, or `%pip` inside notebooks rather than `!pip`). In mixed Conda environments, install `conda-forge` packages *before* `pip` because `pip`'s solver cannot re-solve compiled Conda libraries.
  - Notebooks keep **hidden variable state** when cells execute out of order — always verify finished work via **Restart Kernel and Run All Cells**.

:::{dropdown} Reference table: pip + venv vs uv vs conda / mamba
| Tool | What its solver manages | Spec & lock files | Best research fit |
| --- | --- | --- | --- |
| **`pip` + `venv`** | Python wheels/sdists against current env (no default lockfile) | `pyproject.toml`, `requirements.txt` | Simple pure-Python / wheel-only setups |
| **`uv`** *(used here)* | Python interpreter + `.venv` + fast full-graph PyPI resolver | `pyproject.toml` + version/hash-pinned `uv.lock` | Fast, reproducible project environments |
| **`conda` / `mamba`** | Joint SAT solve across Python + compiled C/Fortran/CUDA libs | `environment.yml` (`conda-lock.yml`) | Binary HPC/astro stacks (`CFITSIO`, `FFTW`, MPI/CUDA) |
:::

## Names, containers and imports

`import numpy as np` loads the `numpy` numerical library under the short alias `np`. A Python `list` (`[...]`) is mutable and uses **0-based indexing**: index `0` is first, `-1` is last, `start:stop` includes `start` while **excluding `stop`**, and `start:stop:step` strides or reverses (`[::2]` every second item, `[::-1]` reversed). A `tuple` (`(...)`) is an **immutable** fixed record supporting unpacking (`lo, hi = (10.0, 20.0)`), a `dict` (`{key: value}`) stores named metadata, `f-strings` format values cleanly (`f"{val:.2f}"`, `f"{val:.2e}"`), and `help(fn)` inspects unfamiliar functions.

In [ ]:
import sys
import numpy as np

print('Interpreter:', sys.executable)
measurements = [1.0, 3.0, 5.0, 7.0]
lo, hi = (10.0, 20.0)  # tuple unpacking (immutable record)
observation = {'object_id': 'mock-01', 'unit': 'km/s'}
val = 1234.567
print('First/last:', measurements[0], measurements[-1], '| bounds:', lo, hi, '| unit:', observation['unit'])
print('Slices [:2], [::2], [::-1]:', measurements[:2], measurements[::2], measurements[::-1])
print(f'Formatted f-strings: {val:.2f} | {val:.2e}')

## Lists vs NumPy arrays and masks

A Python `list` is a general container (`[1, 2] + [3, 4]` concatenates lists instead of adding numbers). `np.array(...)` builds a typed `np.ndarray` supporting fast elementwise arithmetic and boolean masks (`np.nan` marks missing entries, tested with `np.isfinite(...)`; combine array masks using `&` or `|` **with parentheses around each comparison**). Stored **attributes** like `.shape` and `.dtype` take no parentheses, whereas **methods** like `.mean()` and `.sum()` perform a calculation and take `()` — on a 2D `(rows, cols)` array, `axis=0` collapses rows and `axis=1` collapses columns.

![Diagram: 0-based slice bounds and 2D NumPy axis reductions](../book/diagrams/array-axes-slicing.svg)

**Slice bounds and 2D axes:** slice `stop` indices are exclusive, and in 2D NumPy reductions the `axis` argument names the dimension being collapsed ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/array-axes-slicing.dot)).

In [ ]:
values = np.array([1.0, 3.0, np.nan, 7.0])
keep = np.isfinite(values) & (values > 2.0)
print('1D attributes -> shape:', values.shape, '| dtype:', values.dtype)
print('1D kept:', values[keep], '| mean():', values[keep].mean())
assert values[keep].tolist() == [3.0, 7.0]

grid_2d = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # shape (2, 3)
print('2D [0, 1]:', grid_2d[0, 1], '| col 0:', grid_2d[:, 0])
print('axis=0 mean:', grid_2d.mean(axis=0), '| axis=1 sum:', grid_2d.sum(axis=1))
assert grid_2d.mean(axis=0).shape == (3,) and grid_2d.sum(axis=1).shape == (2,)

## Loops, vectorised arrays and functions

`for` loops iterate with 4-space indentation (`enumerate` yields `(index, item)`, and `zip` pairs sequences). List comprehensions (`[f"band-{b}" for b in ('g', 'r', 'i')]`) suit short label or path lists, whereas numerical maths belongs in vectorised NumPy array expressions; `with` blocks close files and temporary folders automatically on exit. Wrap reusable calculations in `def` functions with a concise `"""..."""` docstring (`return` hands back values for code and tests, whereas `print` only displays text), and plot arrays on explicit Matplotlib `fig, ax = plt.subplots(...)` objects.

In [ ]:
def square(value):
    """Return the elementwise square of a scalar or NumPy array."""
    return value ** 2

bands = [f"band-{b}" for b in ('g', 'r', 'i')]
for idx, (band, item) in enumerate(zip(bands, [1.0, 2.0, 3.0])):
    print(f"{idx}: {band} -> {square(item)}")

grid = np.array([1.0, 2.0, 3.0, 4.0])
print('Vectorised array output:', square(grid))
assert square(4.0) == 16.0

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 2.6))
ax.plot(grid, square(grid), 'o-')
ax.set(xlabel='Grid input x', ylabel='square(x)', title='First Figure + Axes plot')
fig.tight_layout()
plt.show()

## Read a traceback

Read Python tracebacks **from the bottom line up**: check the final exception type and message first, then look up to the `---->` arrow marking the line in your code that failed.

```text
Traceback (most recent call last):
  Cell In[4], line 2
    1 fluxes = np.array([10.0, 20.0, 30.0])
----> 2 calibrated = fluxes * np.array([0.9, 1.1])
ValueError: operands could not be broadcast together with shapes (3,) (2,)
```

`ValueError` shows that 1D shapes `(3,)` and `(2,)` do not match on line 2. Check `.shape` on both arrays and supply matching lengths:

In [ ]:
fluxes = np.array([10.0, 20.0, 30.0])
bad_gains = np.array([0.9, 1.1])
try:
    fluxes * bad_gains
except ValueError as exc:
    print(f'{type(exc).__name__}: {exc}')

gains = np.array([0.9, 1.0, 1.1])
assert fluxes.shape == gains.shape == (3,)
calibrated = fluxes * gains
print('Calibrated:', calibrated)

## Views vs `.copy()` in memory

Unlike MATLAB, R or Mathematica — which copy arrays on write — a NumPy slice is a **view** into the same memory buffer, so modifying a slice mutates the original array. Call `.copy()` when you need an independent array.

![Diagram: A NumPy slice shares memory; .copy() allocates an independent array](../book/diagrams/array-views.svg)

**Shared buffer vs independent copy:** slicing aliases `original`; `.copy()` allocates separate memory ([source](https://github.com/xangma/jupyter_data_languages/blob/codex/phd-python-2026/book/diagrams/array-views.dot)).

In [ ]:
original = np.arange(5)
view = original[:2]
view[0] = 99
print('After view[0] = 99     -> original:', original)
assert original[0] == 99

separate = original[:2].copy()
separate[0] = -5
print('After separate[0] = -5 -> original:', original, '| separate:', separate)
assert original[0] == 99

## Your turn

1. Write `positive_finite(values)` returning a 1D array of finite entries strictly greater than zero (`> 0`).
2. Predict its output on `[-2, 0, 1, np.nan, np.inf, 3]`, then verify with `assert` *(hint: begin with `values = np.asarray(values, dtype=float)`).*
3. Explain why replacing missing measurements or failed grid points with `0.0` corrupts a scientific summary.

**Ready for Session 1:** check that you can explain environment lockfiles, `Kernel != terminal`, `.shape` vs `.mean()`, a parenthesised `&` mask, and `return`. Compare with [solutions](../book/solutions.md) once your assertion passes.

In [ ]:
# Define positive_finite(values) and check it with an assert